In [2]:
import os, sys, warnings, pandas as pd, psycopg
from dotenv import load_dotenv
sys.path.insert(0, "..")
from collector.embed import clean_for_embedding        # strips Steam's [b], [url=...] tags for easier reading
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)
def q(sql, **params): return pd.read_sql(sql, conn, params=params)

LABELS = "../data/labels"
TEST_LABEL = f"{LABELS}/patch_test_to_label.csv"
TEST_KEY = f"{LABELS}/patch_test_key.csv"

In [3]:
CUTOFF = "2026-10-23"    # fixed, so a rerun draws the same posts even as new ones arrive
QUOTA = {"none": 30, "keyword_no_tag": 10, "tag": 5, "version_only": 5}
dev = pd.read_csv(f"{LABELS}/patch_sample_key.csv", encoding="utf-8-sig", dtype={"gid": str})

pool = q("""
WITH s AS (
    SELECT f.gid, f.appid, g.name, f.post_date, f.title, n.url, n.contents,
           CASE WHEN f.rule_tag     THEN 'tag'
                WHEN f.rule_keyword THEN 'keyword_no_tag'
                WHEN f.rule_version THEN 'version_only'
                ELSE 'none' END AS stratum
    FROM news_patch_flags f JOIN news n USING (gid) JOIN games g ON g.appid = f.appid
    WHERE f.post_date < %(cutoff)s::date AND NOT (f.gid = ANY(%(dev)s))
), one_per_game AS (
    SELECT *, row_number() OVER (PARTITION BY stratum, appid ORDER BY md5(gid)) AS nth FROM s
)
SELECT *, row_number() OVER (PARTITION BY stratum ORDER BY md5(gid)) AS k
FROM one_per_game WHERE nth = 1
""", cutoff=CUTOFF, dev=dev.gid.tolist())

test = pd.concat([pool[(pool.stratum == s) & (pool.k <= n)] for s, n in QUOTA.items()])
assert len(test) == sum(QUOTA.values()), test.stratum.value_counts()

if os.path.exists(TEST_LABEL) or os.path.exists(TEST_KEY):
    print("test files already exist; not overwriting")
else:
    test = test.sample(frac=1, random_state=20261023)                      # shuffle: no blocks by group
    test["excerpt"] = test.contents.fillna("").map(clean_for_embedding).str[:1500]
    test.sort_values("gid")[["gid", "stratum"]].to_csv(TEST_KEY, index=False, encoding="utf-8-sig")
    (test[["gid", "name", "post_date", "title", "url", "excerpt"]]
         .assign(is_patch_true="", note="")
         .to_csv(TEST_LABEL, index=False, encoding="utf-8-sig"))
    print(f"wrote {len(test)} posts to label")

test files already exist; not overwriting


In [4]:
lab = pd.read_csv(TEST_LABEL, encoding="utf-8-sig",
                  dtype={"gid": str, "note": str, "is_patch_true": "Int64"})
START = 0                                            # 0, 10, 20, 30, 40
for i, r in lab.iloc[START:START + 10].iterrows():
    print(f"[{i}] {r['name']} | {r.post_date} | {r.title}\n     {r.url}\n{r.excerpt}\n{'-' * 100}")

[0] Kingdom Rush 6: Genesis TD | 2026-02-26 | Ask the Kingdom Rush 6 Devs Anything!
     https://steamstore-a.akamaihd.net/news/externalpost/steam_community_announcements/1825727806707604
[p]So… it’s real. Kingdom Rush 6: Genesis is happening.
[/p][p]Making a new Kingdom Rush is wild in the best way. This series has grown so much thanks to you, and we’re ridiculously grateful for that. To celebrate, we’re doing a Dev AMA.[/p][p] On March 3 at 12 PM (GMT-3) , the team will jump into this Steam thread and answer your questions. Development stories? Early lore hints? Design choices? Cut ideas? Random nonsense? Fair game.[/p][p] Drop your questions below and react to your favorites. We’ll answer as many as we can.[/p][p][/p][p]And if you’re ready to rewrite a bit of Linirea’s history with us, wishlist Kingdom Rush 6: Genesis and follow the journey. It really helps more than you think![/p][p][/p][p][dynamiclink href="https://store.steampowered.com/app/4259190/Kingdom_Rush_6_Genesis_TD"][/dy

In [5]:
answers = {
    0: (0, "game and chat announcement"),
    1: (0, "game announcement"),
    2: (1, "bug fixes and balance changes"),
    3: (0, "recap; change announced earlier in its own post"),
    4: (0, "game announcement"),
    5: (1, "added content"),
    6: (1, "new content and bug fixes"),
    7: (0, "sale announcement"),
    8: (0, "game announcement"),
    9: (0, "developer livestream announcement"),
    10: (0, "opt-in open beta branch (rollback branch exists); corrected 2026-10-23: beta rule"),
    11: (0, "participation announcement"),
    12: (1, "content, bug fixes, and balance changes"),
    13: (0, "test build announcement"),
    14: (0, "sale announcement"),
    15: (0, "in-game event announced 3 days ahead; corrected 2026-10-23: further-out schedule rule"),
    16: (1, "bug fixes and quality of life changes"),
    17: (0, "test server announcement"),
    18: (0, "bundle announcement"),
    19: (1, "bug fixes"),
    20: (1, "paid campaign DLC + free new faction for all owners; content"),
    21: (0, "future patch and game announcement"),
    22: (1, "bug fixes and quality of life changes"),
    23: (1, "bug fixes"),
    24: (0, "competition announcement, no game changes"),
    25: (1, "bug fixes"),
    26: (1, "paid DLC + free Update 11 for all players; content + bugfix"),
    27: (0, "merch announcement, no game changes"),
    28: (0, "season update announced 5 days ahead; corrected 2026-10-23: further-out schedule rule"),
    29: (1, "hotfix"),
    30: (1, "content update"),
    31: (0, "trailer announcement"),
    32: (1, "scheduled next day, changes listed; bugfix + content; corrected 2026-10-23: same-day rule"),
    33: (0, "update trailer announcement"),
    34: (0, "thank you announcement"),
    35: (0, "thank you announcement"),
    36: (0, "launch of a separate mobile game, rewards are for The Division 2; corrected 2026-10-23"),    37: (1, "content, bug fixes"),
    38: (0, "demo release announcement"),
    39: (1, "hotfix"),
    40: (1, "content and bug fixes"),
    41: (1, "flash update, scheduled same day, changes listed; content"),
    42: (0, "lore announcement"),
    43: (0, "merch announcement"),
    44: (0, "opt-in development branch; mostly free DLC preview; main game unchanged"),    45: (1, "bug fixes and improvements"),
    46: (0, "art announcement"),
    47: (0, "merch announcement"),
    48: (0, "free weekend announcement"),
    49: (1, "content and event announcement")
}

done = sorted(answers)
assert all(v[0] in (0, 1) for v in answers.values())
lab.loc[done, "is_patch_true"] = [answers[i][0] for i in done]
lab.loc[done, "note"] = [answers[i][1] for i in done]
lab[["gid", "name", "post_date", "title", "url", "excerpt", "is_patch_true", "note"]].to_csv(
    TEST_LABEL, index=False, encoding="utf-8-sig")
print(f"{len(done)} of {len(lab)} labeled")

50 of 50 labeled


In [31]:
import json, time
from typing import Literal
from pydantic import BaseModel
from google import genai
from google.genai import types, errors

MODEL = "gemini-3.6-flash"                   # the Flash model you picked in AI Studio
PROMPT_VERSION = "v1"
SYSTEM = open(f"../prompts/patch_{PROMPT_VERSION}.txt", encoding="utf-8").read()
CACHE = f"../data/gemini/posts_{PROMPT_VERSION}_{MODEL}.jsonl"
RPM = 5
SLEEP = 60 / RPM * 1.1                          # seconds between calls: 60 / your free-tier requests per minute
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

BATCH = 20                         # posts per call: the daily limit is 20 calls, so calls are what we save

class PostLabel(BaseModel):
    post: int                      # the number from "=== POST n ===", mapped back to the gid below
    reason: str                    # first, so the model explains before it decides
    is_patch: bool
    patch_type: list[Literal["bugfix", "balance", "content", "technical", "server-side"]]

class BatchLabels(BaseModel):
    labels: list[PostLabel]

def post_text(r):
    body = clean_for_embedding(r.contents or "")[:4000]
    return f"Game: {r.name}\nPosted: {r.post_date}\nTitle: {r.title}\n\n{body}"

def batch_text(chunk):
    return "\n\n".join(f"=== POST {n} ===\n{post_text(r)}" for n, r in enumerate(chunk.itertuples(), 1))

def read_cache():
    if not os.path.exists(CACHE) or os.path.getsize(CACHE) == 0:
        return pd.DataFrame(columns=["gid", "reason", "is_patch", "patch_type"])
    return pd.read_json(CACHE, lines=True, dtype={"gid": str})

def label_posts(df):
    """Label posts in batches, once per prompt version. Answers are cached, so a rerun only sends posts not yet labeled."""
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    todo = df[~df.gid.isin(set(read_cache().gid))]
    calls = 0
    failed_in_row = 0
    with open(CACHE, "a", encoding="utf-8") as f:
        for start in range(0, len(todo), BATCH):
            chunk = todo.iloc[start:start + BATCH]
            for attempt in range(1):
                try:
                    resp = client.models.generate_content(
                        model=MODEL, contents=batch_text(chunk),
                        config=types.GenerateContentConfig(
                            system_instruction=SYSTEM, temperature=0,
                            response_mime_type="application/json", response_schema=BatchLabels))
                    calls += 1
                    break
                except errors.APIError as e:
                    if e.code == 429 and "PerDay" in str(e):     # daily quota: retrying today is pointless
                        print(f"Daily limit reached after {calls} calls; rerun after midnight Pacific.")
                        return read_cache()
                    if e.code not in (429, 500, 503):             # permanent: bad key, model or request
                        raise
                    print(f"{e.code}, retrying")
                    time.sleep(90 * (attempt + 1))
            else:                                                 # all retries failed for this batch
                failed_in_row += 1
                if failed_in_row >= 2:                            # two batches in a row: an outage, not bad luck
                    print("Gemini looks unavailable; stopping. Rerun later; nothing is lost.")
                    return read_cache()
                continue
            failed_in_row = 0                                          # gave up on this batch; a rerun retries it
            if resp.parsed is None:
                print(f"unparseable answer: {resp.text[:200]!r}")
                continue
            gids = dict(enumerate(chunk.gid, 1))
            got = {l.post: l for l in resp.parsed.labels if l.post in gids}
            if len(got) < len(chunk):
                print(f"{len(chunk) - len(got)} posts missing from the answer; a rerun will retry them")
            for n, l in got.items():
                row = l.model_dump(exclude={"post"}) | {"gid": gids[n], "model": MODEL,
                                                        "prompt_version": PROMPT_VERSION, "batch": BATCH}
                f.write(json.dumps(row) + "\n")
            f.flush()
            time.sleep(SLEEP)
    print(f"{calls} calls this run")
    return read_cache()

In [24]:
dev_lab = pd.read_csv(f"{LABELS}/patch_posts_to_label.csv", encoding="utf-8-sig", dtype={"gid": str})
EXAMPLE_TITLES = ["Minor Update - Main branch", "EA SPORTS FC™ 27 | Closed Beta Feedback Update",
                  "Demo Update 1.2 - Drag and then also Drop", "New maps in workshop", "v0.4.3 Open Beta"]
examples = dev_lab.title.isin(EXAMPLE_TITLES) | dev_lab.title.str.startswith("1.4.3.1 Known Issues")
assert examples.sum() == 6, dev_lab.title[examples]
dev_eval = dev_lab[~examples]

posts = q("""SELECT n.gid, g.name, (n.posted_at AT TIME ZONE 'UTC')::date AS post_date, n.title, n.contents
             FROM news n JOIN games g USING (appid) WHERE n.gid = ANY(%(gids)s)""", gids=dev_eval.gid.tolist())
res = label_posts(posts)

cmp = dev_eval.merge(res, on="gid", validate="one_to_one")
cmp["agree"] = cmp.is_patch_true == cmp.is_patch.astype(int)
print(f"{PROMPT_VERSION}: {cmp.agree.sum()} of {len(cmp)} agree")
cmp.loc[~cmp.agree, ["title", "is_patch_true", "note", "is_patch", "reason"]]

503, retrying
1 calls this run
v1: 33 of 34 agree


,title,is_patch_true,note,is_patch,reason
29,It's Time to Celebrate Sky's 7th Anniversary!,1,"in-game event activities live (dev-run, not ju...",False,This post announces a community anniversary ev...


In [28]:
import math
def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - h), min(1.0, c + h)

test_lab = pd.read_csv(TEST_LABEL, encoding="utf-8-sig", dtype={"gid": str, "is_patch_true": "Int64"})
test_key = pd.read_csv(TEST_KEY, encoding="utf-8-sig", dtype={"gid": str})
test_posts = q("""SELECT n.gid, g.name, (n.posted_at AT TIME ZONE 'UTC')::date AS post_date, n.title, n.contents
                  FROM news n JOIN games g USING (appid) WHERE n.gid = ANY(%(gids)s)""", gids=test_lab.gid.tolist())
res = label_posts(test_posts)

t = (test_lab.merge(test_key, on="gid", validate="one_to_one")
             .merge(res, on="gid", validate="one_to_one"))
t["gemini"] = t.is_patch.astype(int)
t["rule"] = (t.stratum != "none").astype(int)          # the D9 rule flags every group except 'none'
for who in ["rule", "gemini"]:
    tp = ((t[who] == 1) & (t.is_patch_true == 1)).sum()
    fp = ((t[who] == 1) & (t.is_patch_true == 0)).sum()
    fn = ((t[who] == 0) & (t.is_patch_true == 1)).sum()
    plo, phi = wilson(tp, tp + fp); rlo, rhi = wilson(tp, tp + fn)
    print(f"{who:6}  agree {(t[who] == t.is_patch_true).sum()}/{len(t)}   "
          f"precision {tp}/{tp + fp} ({plo:.0%}-{phi:.0%})   recall {tp}/{tp + fn} ({rlo:.0%}-{rhi:.0%})")
t.loc[t.gemini != t.is_patch_true, ["title", "stratum", "is_patch_true", "note", "gemini", "reason"]]

503, retrying
1 calls this run
rule    agree 40/50   precision 15/20 (53%-89%)   recall 15/20 (53%-89%)
gemini  agree 49/50   precision 20/21 (77%-99%)   recall 20/20 (84%-100%)


,title,stratum,is_patch_true,note,gemini,reason
42,Another wave of rumors,none,0,lore announcement,1,This post announces a new in-game event and lo...


In [23]:
lab = pd.read_csv(TEST_LABEL, encoding="utf-8-sig", dtype={"gid": str, "is_patch_true": "Int64"})
lab[lab.note.str.contains("beta|schedul|DLC|event|early access|demo|test", case=False, na=False)][["title", "post_date", "is_patch_true", "note"]]

,title,post_date,is_patch_true,note
10,"Stellaris 4.3 ""Cetus"" Open Beta Updated (2026-...",2026-03-04,0,opt-in open beta branch (rollback branch exist...
13,NOW TESTING: Steam test build for 9.6.1 issues,2026-05-06,0,test build announcement
15,Celebrate First Contact Day!,2026-04-04,0,in-game event announced 3 days ahead; correcte...
17,Test Server launch [04.05.26],2026-05-04,0,test server announcement
20,The Veti's Wrath Expansion is available NOW!,2026-09-29,1,paid campaign DLC + free new faction for all o...
26,Planet Coaster 2 | Silver Screen Pack - Out Now!,2026-08-19,1,paid DLC + free Update 11 for all players; con...
28,Call of Duty: Black Ops 7 and Call of Duty: Wa...,2026-03-06,0,season update announced 5 days ahead; correcte...
32,Update 0.29.1 Patch Notes and Known Issues,2025-12-01,1,"scheduled next day, changes listed; bugfix + c..."
38,SPRAWL Zero: The Demo is Live!,2026-06-09,0,demo release announcement
41,Flash Update on August 5,2026-08-05,1,"flash update, scheduled same day, changes list..."


In [33]:
CUTOFF = "2026-10-23"          # same fixed cutoff as the test set, so the post set doesn't grow
priority = q("""
WITH untagged AS (
    SELECT appid FROM news_patch_flags
    WHERE post_date < %(cutoff)s::date
    GROUP BY appid HAVING count(*) FILTER (WHERE rule_tag) = 0
)
SELECT n.gid, g.name, (n.posted_at AT TIME ZONE 'UTC')::date AS post_date, n.title, n.contents
FROM news n JOIN games g USING (appid) JOIN untagged u USING (appid)
WHERE (n.posted_at AT TIME ZONE 'UTC')::date < %(cutoff)s::date
ORDER BY md5(n.gid)            -- random order, so a partial run is a fair sample, not a few games
""", cutoff=CUTOFF)

res = label_posts(priority)
done = priority.gid.isin(res.gid).sum()
print(f"priority posts: {done:,} of {len(priority):,} labeled, from {priority.name.nunique()} games")

503, retrying
503, retrying
503, retrying
Gemini looks unavailable; stopping. Rerun later; nothing is lost.
priority posts: 192 of 1,162 labeled, from 95 games


In [14]:
resp = client.models.generate_content(model=MODEL, contents="Reply with OK.")
print(resp.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [16]:
skip = ("tts", "image", "audio", "live", "embedding", "preview", "exp", "vision", "native", "thinking")
for m in client.models.list():
    name = m.name.lower()
    if "generateContent" in (m.supported_actions or []) and "flash" in name and not any(s in name for s in skip):
        print(m.name, "|", m.display_name)

models/gemini-2.5-flash | Gemini 2.5 Flash
models/gemini-flash-latest | Gemini Flash Latest
models/gemini-flash-lite-latest | Gemini Flash-Lite Latest
models/gemini-2.5-flash-lite | Gemini 2.5 Flash-Lite
models/gemini-3.1-flash-lite | Gemini 3.1 Flash Lite
models/gemini-3.5-flash | Gemini 3.5 Flash
models/gemini-3.5-flash-lite | Gemini 3.5 Flash Lite
models/gemini-omni-1.1-flash | Gemini Omni 1.1 Flash
models/gemini-3.6-flash | Gemini 3.6 Flash
models/gemini-3.7-flash | Gemini 3.7 Flash
models/gemini-3.8-flash | Gemini 3.8 Flash


In [19]:
resp = client.models.generate_content(model="gemini-3.6-flash", contents="Reply with OK.")
print(resp.text)

OK.
